```bash
pip install tf2onnx
python -c "import tf2onnx, tensorflow as tf; m=tf.keras.models.load_model('q40/model'); tf2onnx.convert.from_keras(m, output_path='model.onnx')"
```

In [3]:
:dep tract-onnx = { version = "0.21.8" }
:dep tract-core = { version = "0.21.8" }
:dep ndarray = "0.15"
:dep image = "0.24"
:dep rand = { version = "0.9", features = ["std"] }

use tract_onnx::prelude::*;
use rand::Rng;

In [5]:
// ========== Adam optimizer ==========
struct Adam {
    lr: f32, beta1: f32, beta2: f32, eps: f32,
    m: Vec<f32>, v: Vec<f32>, t: i32,
}
impl Adam {
    fn new(size: usize, lr: f32) -> Self {
        Adam { lr, beta1: 0.9, beta2: 0.999, eps: 1e-8,
               m: vec![0.0; size], v: vec![0.0; size], t: 0 }
    }
    fn step(&mut self, params: &mut Vec<f32>, grads: &[f32]) {
        self.t += 1;
        let lr_t = self.lr
            * (1.0 - self.beta2.powi(self.t)).sqrt()
            / (1.0 - self.beta1.powi(self.t));
        for i in 0..params.len() {
            self.m[i] = self.beta1 * self.m[i] + (1.0 - self.beta1) * grads[i];
            self.v[i] = self.beta2 * self.v[i] + (1.0 - self.beta2) * grads[i].powi(2);
            params[i] -= lr_t * self.m[i] / (self.v[i].sqrt() + self.eps);
            params[i] = params[i].clamp(0.0, 1.0);
        }
    }
}

// ========== 型エイリアス（長い型名を短縮）==========
type Model = SimplePlan<TypedFact, Box<dyn TypedOp>, Graph<TypedFact, Box<dyn TypedOp>>>;

fn run_model(model: &Model, pixels: &[f32]) -> TractResult<f32> {
    let arr = tract_ndarray::Array4::from_shape_vec(
        (1, 32, 32, 1), pixels.to_vec()
    )?;
    let tensor: Tensor = arr.into();
    let result = model.run(tvec![tensor.into()])?;
    let probs = result[0].to_array_view::<f32>()?;
    Ok(probs[[0, 10]])
}

fn numerical_grad(model: &Model, pixels: &[f32], eps: f32) -> TractResult<Vec<f32>> {
    let f0 = run_model(model, pixels)?;
    let mut grads = vec![0.0f32; pixels.len()];
    for i in 0..pixels.len() {
        let mut p = pixels.to_vec();
        p[i] = (p[i] + eps).min(1.0);
        let f1 = run_model(model, &p)?;
        grads[i] = -(f1 - f0) / eps;  // loss = -flag_prob
    }
    Ok(grads)
}

fn main() -> TractResult<()> {
    let model = tract_onnx::onnx()
        .model_for_path("model.onnx")?
        .with_input_fact(0, f32::fact(&[1, 32, 32, 1]).into())?
        .into_optimized()?
        .into_runnable()?;

    // rand 0.9 では random() を使う（gen は予約語）
    let mut rng = rand::rng();
    let mut pixels: Vec<f32> = (0..1024).map(|_| rng.random::<f32>()).collect();

    let mut adam = Adam::new(1024, 0.01);

    for step in 0..=3000usize {
        let flag_prob = run_model(&model, &pixels)?;
        if step % 500 == 0 {
            println!("Step {:4}: flag prob = {:.1}%", step, flag_prob * 100.0);
        }
        if flag_prob > 0.999 {
            println!("Converged at step {}!", step);
            break;
        }
        let grads = numerical_grad(&model, &pixels, 1e-3)?;
        adam.step(&mut pixels, &grads);
    }

    // 画像保存
    let img_data: Vec<u8> = pixels.iter().map(|&p| (p * 255.0) as u8).collect();
    image::GrayImage::from_raw(32, 32, img_data)
        .expect("image error")
        .save("recovered_flag_rust.png")
        .expect("save error");
    println!("Saved: recovered_flag_rust.png");
    Ok(())
}
main()

Step    0: flag prob = 2.7%
Step  500: flag prob = 99.8%
Step 1000: flag prob = 99.9%
Step 1500: flag prob = 99.9%
Step 2000: flag prob = 99.9%
Step 2500: flag prob = 99.9%
Step 3000: flag prob = 99.9%
Saved: recovered_flag_rust.png


Ok(())